# LLM and literature
The goal of our project was to test the performance of large language models on the literature contained in *Project Gutenberg*, with special regards of differences in the language accuracies. We tested how different challenges changed the accuracy of the model. Accuracy in this case meant the capability of the model to guess from which book a certain text slice was. 

For that we first needed our basic programm to use for the different tests. 

## Basics

### Used Libraries

In [1]:
import os
import random
import re
import csv
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt

### Fixed Variables
These are variables that are used, at several points. When changed the whole code should be executed again, as they influence all following parts. The current values were chosen to be a balance between performance and sufficient results. 

In [11]:
NUM_TESTED_BOOKS = 100
SNIPPET_LENGTH = 1000
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
DATASET_NAME = "manu/project_gutenberg"
OUTPUT_FILE = "book_reasoning.txt"

### Important Methods
Methods that are used repeatedly throughout the program, optional parameters should make them more flexible, avoiding to use several similar ones. 

First one is to read the dataset, in the given language and import the books from there. Books that are translations are being excluded. 

In [6]:
def is_a_translation(text):
    translator_match = re.search(r"Translator:\s*(.*)", text, re.IGNORECASE)
    return translator_match is not None


def load_dataset(language):
    books = []
    dataset = load_dataset(DATASET_NAME, split=language, streaming=True)

    iterator = iter(dataset)
    
    while len(books) < target_count:
        try: 
            item = next(iterator)
            text = item["text"]
            title = "Unknown Title"

            # Skips translations
            if is_a_translation(text) is true:
                continue
            
            # Searches for "Title" in the header
            title_match = re.search(
                r"Title:\s*(.*?)(?=\r?\n\w+:|\r?\n\r?\n)",
                text,
                re.IGNORECASE | re.DOTALL,
            )

            if title_match: 
                title = re.sub(r"\s+", " ", title_match.group(1)).strip()
                title = title.lower()

            # Skips books with recurring titles or no title
            if title == "Unknown Title" or title in seen_titles:
                continue

            snippet_start = len(text) // 2
            snippet_end = snippet_start + snippet_length
            snippet = text[snippet_start:snippet_end].strip()

            seen_titles.add(title)
            books.append({"title":title, "text":snippet})

        except StopIteration:
            print("Reached end of dataset.")
            break
    return books


Next a method to create the multiple choice questions that the model gets asked. The wrong answers coming from the pool of possible titles from the database. 

In [7]:
def create_multiple_choice_questions(books):
    question_set = []
    all_titles = [book["title"] for book in books]

    for book in books:
        correct_title = book["title"]

        wrong_choices = [t for t in all_titles if t != correct_title]

        if len(wrong_choices) < 3:
            raise ValueError("Not enough titles to create questions")
        
        choices = random.sample(wrong_choices, 3) + [correct_title]
        random.shuffle(choices)

        correct_letter = chr(65 + choices.index(correct_title))

        question_set.append({"snippet": book["text"], "choices": choices, "correct_letter": correct_letter, "correct_title": correct_title})

    return question_set

In [13]:
def testing_accuracy(model, tokenizer, question_set, reasoning = False):
    accuracy = []
    correct_predictions = 0
    failed_parses = 0
    incorrect_predictions = 0

    for i, item in enumerate(question_set):
        prompt_choices = "\n".join([
        f"{chr(65 + idx)}) {choice}"
        for idx, choice in enumerate(item["choices"])
        ])

        if(reasoning):
            prompt = ("You are a helpful literature assistent. Your task is to identify to which book a text snippet belongs to"
                "Structure your response exactly like this. Do NOT use any other formatting: \n"
                "Answer: [Single answer choice: A, B, C or D]\n"
                "Explanation: [A brief 1-2 sentence explanation, why that title is the best match for the book]"
                f"Text Snippet:\n\"\"\"\n{item['snippet']}\n\"\"\"\n\n"
                f"Which book is this snippet from?\n{prompt_choices}\n\n"
                "Provide your answer using the correct format."
                )
        else:        
            prompt = ("You are a helpful literature assistent. Your task is to identify to which book a text snippet belongs to"
                    "Respond only (!) with the single letter choice (A, B, C or D), corresponding to the correct book."
                    "Do not output anything else."
                    f"\n\"\"\n{item['snippet']}\n\"\"\n\n"
                    f"Which book is this snippet from?\n{prompt_choices}\n\n"
                    "Answer (A/B/C/D):"
                    )
        
        text_input = tokenizer.apply_chat_template(
            prompt, 
            tokenize=True,
            add_generation_prompt = True,
        )

        model_inputs = tokenizer([text_input], return_tensors="pt").to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=4,
                temperature=0.1,
                do_sample=False,
            )
        
        generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip().upper()

        predicted_letter_match = re.search(r"[A-D]", response)
        
        if predicted_letter_match:
            predicted_letter = predicted_letter_match.group(0)
        else:
            predicted_letter = "Unknown"
            failed_parses += 1

        if predicted_letter == item["correct_letter"]:
            correct_predictions += 1
        else: 
            incorrect_predictions += 1

        if reasoning:
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f_out:
                f_out.write(f"--- Book {i+1}/{len(subset_evaluation_set)} ---\n")
                f_out.write(f"Correct Answer:  [{item['correct_letter']}] {correct_title}\n")
                f_out.write(f"Model Predicted: [{predicted_letter}] {predicted_title} ({'PASS' if is_correct else 'FAIL'})\n")
                f_out.write(f"Model Response:\n{full_response}\n")
                f_out.write("-" * 40 + "\n\n")
                f_out.flush()

        accuracy.append({"correct":correct_predictions, "prompt_errors": failed_parses, "incorrect":incorrect_predictions})
        return accuracy

For later tests we are going to add noise into the text. For this the following method can be used.

In [ ]:
def add_noise(books, percentage):
    if percentage <= 0:
        return books
    text_snippets = [book["text"] for book in books]

    for t in text_snippets:
        text_chars = list(t)
        num_chars_to_change = int(len(text_chars) * (percentage / 100.0))
        indices_to_replace = random.sample(range(len(text_chars)), num_chars_to_change)
        random_pool = string.ascii_letters 
        for idx in indices_to_replace:
            # Replace current character with a random letter
            text_chars[idx] = random.choice(random_pool)
        t = "".join(text_chars)
    
    



    